# 02. LoRA и новые векторы
Выбери T4. Укажи те же имя и признак, что в 01. Код берётся из коммита, записанного в запуске.

In [ ]:
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO = "https://github.com/danilov-e/lora-activation-steering.git"
RUN_NAME = "research_api_01"
TRAIT = "sycophancy"  # evil / sycophancy / hallucination
GIT_REF = "main"      # используется при создании нового запуска

SOURCE = Path("/content/lora-steering-bootstrap")
if not SOURCE.exists():
    run(["git", "clone", "--quiet", "--depth", "1", REPO, str(SOURCE)], check=True)

colab = run_path(str(SOURCE / "notebooks" / "colab.py"))
PROJECT, RUNS, COMMIT = colab["setup"](SOURCE, RUN_NAME, TRAIT, GIT_REF, resume=True)

from src import judge as j, report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

In [ ]:
RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
CONFIG = w.run_info(RUN)["config"]
choices = w.load_choices(RUN)
print(RUN)
display(pd.DataFrame(choices))

## Обучение
Обычная LoRA обучается на neg. Для каждой настройки учится отдельная steered LoRA с замороженным исходным вектором. Новые векторы извлекаются с выключенным стирингом.

In [ ]:
labels = w.train_all(RUN)
display(Image(filename=str(r.validation_loss(RUN, labels))))

## Нормы и косинусное сходство
Файлы геометрии и графики сохраняются вместе с векторами.

In [ ]:
for label in labels:
    geometry, plot = r.compare_vectors(RUN, label)
    print(label)
    display(geometry.loc[geometry.layer.isin([c["layer"] for c in choices])].round(4))
    display(Image(filename=str(plot)))